1. Importazione delle librerie necessarie

In [50]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.base import BaseEstimator, ClassifierMixin
import matplotlib.pyplot as plt

# Caricamento del dataset
attributes = ["Status_account", "Duration", "Credit_history", "Purpose",
              "Credit_amount", "Savings_account_bonds", "Employment_since", "Install_rate",
              "Personal_status_sex", "debtors_guarantors", "Present_residence_since",
              "Property", "Age", "Other_install_plans", "Housing", "No_credits", "Job",
              "No_people_liable", "Telephone", "foreign_worker", "good_bad"]

df = pd.read_csv("statlog+german+credit+data/german.data", sep=' ', names=attributes)

# Divisione in feature e target
X_not_encoded, X_test_not_encoded, y, y_test = train_test_split(
    df.drop(columns="good_bad"), df['good_bad'], stratify=df['good_bad'], train_size=0.8, random_state=15)

# Encoding
to_remove = ["Duration", "Credit_amount", "Install_rate", "Present_residence_since", "Age", "No_credits", "No_people_liable"]
to_be_encoded = [col for col in attributes if col not in to_remove and col != "good_bad"]

X_encoded = pd.get_dummies(X_not_encoded, columns=to_be_encoded, dtype=int)
X_test_encoded = pd.get_dummies(X_test_not_encoded, columns=to_be_encoded, dtype=int)

# Normalizzazione
scaler = MinMaxScaler()
X = scaler.fit_transform(X_encoded)
X_test = scaler.transform(X_test_encoded)

# Conversione del target
y = np.where(y == 2, 0, 1)
y_test = np.where(y_test == 2, 0, 1)


3. Definizione del modello e della classe personalizzata per la ricerca iperparametrica

In [51]:
# Definizione del modello
class Model_classification_multi(nn.Module): 
    def __init__(self, in_features, out_features, hidden_1, hidden_2, hidden_3):
        super(Model_classification_multi, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden_1),
            nn.ReLU(),
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),
            nn.Linear(hidden_2, hidden_3),
            nn.ReLU(),
            nn.Linear(hidden_3, out_features)
        )
    
    def forward(self, x):
        return F.softmax(self.layers(x), dim=1)

class PyTorchClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, in_features, hidden_1, hidden_2, hidden_3, lr=0.001, epochs=50, batch_size=64):
        self.in_features = in_features
        self.hidden_1 = hidden_1
        self.hidden_2 = hidden_2
        self.hidden_3 = hidden_3
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.model = Model_classification_multi(in_features, 2, hidden_1, hidden_2, hidden_3)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        self.loss_fn = nn.CrossEntropyLoss()

    def fit(self, X, y):
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.long)
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        
        for epoch in range(self.epochs):
            self.model.train()
            for batch_X, batch_y in loader:
                y_pred = self.model(batch_X)
                loss = self.loss_fn(y_pred, batch_y)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
        return self

    def predict(self, X):
        self.model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32)
        with torch.no_grad():
            y_pred = self.model(X_tensor)
        return torch.argmax(y_pred, dim=1).numpy()

    def predict_proba(self, X):
        self.model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32)
        with torch.no_grad():
            y_pred = self.model(X_tensor)
        return y_pred.numpy()

4. Ricerca degli Iperparametri con GridSearchCV

In [53]:
# Definizione del Grid per la ricerca degli iperparametri
param_grid = {
    'hidden_1': [64, 128],
    'hidden_2': [32, 64],
    'hidden_3': [16, 32],
    'lr': [0.001, 0.0001],
    'batch_size': [32, 64],
    'epochs': [50, 100]
}

Continuiamo la configurazione della ricerca iperparametrica e dell'analisi delle tre configurazioni con l'implementazione di StratifiedKFold:

Continuazione della configurazione di GridSearchCV con StratifiedKFold

In [54]:
from sklearn.model_selection import StratifiedKFold

# Implementazione di StratifiedKFold per preservare la distribuzione delle classi durante la ricerca
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=15)

# Inizializzazione del modello PyTorchClassifier
model = PyTorchClassifier(in_features=X.shape[1], hidden_1=64, hidden_2=32, hidden_3=16)

# Implementazione della ricerca degli iperparametri con GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_search.fit(X, y)

# Migliori iperparametri
print(f"Migliori iperparametri trovati: {grid_search.best_params_}")


/home/mrnbd/.local/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:993: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/home/mrnbd/.local/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 982, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/home/mrnbd/.local/lib/python3.10/site-packages/sklearn/metrics/_scorer.py", line 253, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "/home/mrnbd/.local/lib/python3.10/site-packages/sklearn/metrics/_scorer.py", line 345, in _score
    y_pred = method_caller(
  File "/home/mrnbd/.local/lib/python3.10/site-packages/sklearn/metrics/_scorer.py", line 87, in _cached_call
    result, _ = _get_response_values(
  File "/home/mrnbd/.local/lib/python3.10/site-packages/sklearn/utils/_response.py", line 198, in _ge

Migliori iperparametri trovati: {'batch_size': 32, 'epochs': 50, 'hidden_1': 64, 'hidden_2': 32, 'hidden_3': 16, 'lr': 0.001}


5. Valutazione delle tre configurazioni
- Senza PCA e senza SMOTE

In [55]:
# Utilizzo del modello con i migliori iperparametri per la valutazione
best_model = grid_search.best_estimator_

# Previsione
y_pred = best_model.predict(X_test)

# Confusion Matrix e Classification Report
print("Senza PCA e senza SMOTE:")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")



Senza PCA e senza SMOTE:
[[ 27  33]
 [ 18 122]]
              precision    recall  f1-score   support

           0       0.60      0.45      0.51        60
           1       0.79      0.87      0.83       140

    accuracy                           0.74       200
   macro avg       0.69      0.66      0.67       200
weighted avg       0.73      0.74      0.73       200

Accuracy: 0.7450
